# Параметрическая анимация модели SIR на сетях Петри

**Цель:** Исследовать влияние параметров β и γ на динамику эпидемии
с помощью анимации.

**Автор:** Чувакина Мария Владимировна
**Дата:** 2026-04-21

## Теоретическое введение

Параметрическая анимация позволяет наглядно сравнить, как разные
значения β и γ влияют на распространение эпидемии.

- **β** — скорость заражения (чем выше, тем быстрее растёт I)
- **γ** — скорость выздоровления (чем выше, тем быстрее падает I)

## Исследуемые параметры

In [1]:
using DrWatson
@quickactivate

using Plots
using DataFrames

include(srcdir("SIRPetri.jl"))
using .SIRPetri

Набор параметров для исследования

In [2]:
params_list = [
    (β = 0.2, γ = 0.1, name = "β=0.2, γ=0.1 (слабая эпидемия)"),
    (β = 0.3, γ = 0.1, name = "β=0.3, γ=0.1 (средняя эпидемия)"),
    (β = 0.5, γ = 0.1, name = "β=0.5, γ=0.1 (сильная эпидемия)"),
    (β = 0.3, γ = 0.2, name = "β=0.3, γ=0.2 (быстрое выздоровление)"),
    (β = 0.5, γ = 0.2, name = "β=0.5, γ=0.2 (комбинированный)"),
]

tmax = 100.0

println("="^60)
println("ПАРАМЕТРИЧЕСКАЯ АНИМАЦИЯ МОДЕЛИ SIR")
println("="^60)

ПАРАМЕТРИЧЕСКАЯ АНИМАЦИЯ МОДЕЛИ SIR


## Создание анимаций для каждого набора параметров

In [3]:
for (idx, params) in enumerate(params_list)
    β = params.β
    γ = params.γ
    name = params.name

    println("Создание анимации для: $name")

    net, u0, states = build_sir_network(β, γ)
    df = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.2, rates = [β, γ])

    anim = @animate for i in 1:length(df.time)
        bar(
            ["S", "I", "R"],
            [df.S[i], df.I[i], df.R[i]],
            ylims = (0, 1000),
            title = "$name | Время = $(round(df.time[i], digits=1))",
            ylabel = "Численность",
            color = [:blue, :red, :green],
            legend = false,
        )
    end

    filename = "sir_animation_β=$(β)_γ=$(γ).gif"
    gif(anim, plotsdir(filename), fps = 10)
    println("  Сохранено: plots/$filename")
end

Создание анимации для: β=0.2, γ=0.1 (слабая эпидемия)


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_β=0.2_γ=0.1.gif


  Сохранено: plots/sir_animation_β=0.2_γ=0.1.gif
Создание анимации для: β=0.3, γ=0.1 (средняя эпидемия)
  Сохранено: plots/sir_animation_β=0.3_γ=0.1.gif
Создание анимации для: β=0.5, γ=0.1 (сильная эпидемия)


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_β=0.3_γ=0.1.gif


  Сохранено: plots/sir_animation_β=0.5_γ=0.1.gif
Создание анимации для: β=0.3, γ=0.2 (быстрое выздоровление)


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_β=0.5_γ=0.1.gif


  Сохранено: plots/sir_animation_β=0.3_γ=0.2.gif
Создание анимации для: β=0.5, γ=0.2 (комбинированный)


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_β=0.3_γ=0.2.gif


  Сохранено: plots/sir_animation_β=0.5_γ=0.2.gif


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_β=0.5_γ=0.2.gif


## Сравнительная анимация

In [4]:
println()
println("Создание сравнительной анимации...")

cases = [
    (β = 0.2, γ = 0.1, color = :blue, label = "β=0.2, γ=0.1"),
    (β = 0.3, γ = 0.1, color = :red, label = "β=0.3, γ=0.1"),
    (β = 0.5, γ = 0.1, color = :green, label = "β=0.5, γ=0.1"),
]

dfs = []
for case in cases
    net, u0, states = build_sir_network(case.β, case.γ)
    df = simulate_deterministic(net, u0, (0.0, tmax), saveat = 0.2, rates = [case.β, case.γ])
    push!(dfs, (df = df, case = case))
end

anim_compare = @animate for i in 1:length(dfs[1].df.time)
    p = plot(
        title = "Сравнение динамики I(t) при разных β",
        xlabel = "Группа",
        ylabel = "Численность",
        ylims = (0, 1000),
        legend = :topleft,
    )

    for (j, data) in enumerate(dfs)
        bar!(
            p, [data.case.label],
            [data.df.I[i]],
            color = data.case.color,
            alpha = 0.7,
            label = data.case.label,
        )
    end

    annotate!(p, 0.5, 950, text("Время = $(round(dfs[1].df.time[i], digits=1))", 10))
end

gif(anim_compare, plotsdir("sir_animation_comparison.gif"), fps = 10)
println("Сравнительная анимация сохранена: plots/sir_animation_comparison.gif")


Создание сравнительной анимации...
Сравнительная анимация сохранена: plots/sir_animation_comparison.gif


[ Info: Saved animation to /afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab06/project/plots/sir_animation_comparison.gif


## Выводы

1. **Влияние β:** чем выше β, тем быстрее растёт I и выше пик
2. **Влияние γ:** чем выше γ, тем быстрее I падает после пика
3. **Комбинированный эффект:** при высоких β и γ эпидемия протекает быстрее

In [5]:
println()
println("="^60)
println("ПАРАМЕТРИЧЕСКАЯ АНИМАЦИЯ ЗАВЕРШЕНА")
println("="^60)


ПАРАМЕТРИЧЕСКАЯ АНИМАЦИЯ ЗАВЕРШЕНА
